# YOLOv10n -> ONNX 변환

1차 학습된 `hagrid_stage1_8cls_best.pt`를 ONNX로 변환

`nms=True`로 export하여 NMS(중복 박스 제거)까지 모델 안에 포함

## 0. 환경 설정

In [1]:
!pip install ultralytics onnx onnxruntime onnxslim

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 73.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 238.4/238.4 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 2.9 MB/s eta 0:00:00


## 1. Drive 마운트 및 가중치 로드

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# ============================================================
# CONFIG
# ============================================================

# 변환할 가중치 경로 (1차 학습 결과 -> 2차 학습 결과 변환 시 파일명 수정)
WEIGHTS_PATH = "/content/drive/MyDrive/HandSpark/weights/hagrid_stage1_8cls_best.pt"

# 추론 시 사용할 입력 이미지 크기 (학습 때와 동일하게 640)
IMG_SIZE = 640

# 변환된 ONNX를 Drive에 저장할 경로
DRIVE_SAVE_DIR = "/content/drive/MyDrive/HandSpark/weights"

# ============================================================

## 2. ONNX 변환 (NMS 포함)

In [4]:
from ultralytics import YOLO

model = YOLO(WEIGHTS_PATH)
print("클래스 목록:", model.names)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
클래스 목록: {0: 'palm', 1: 'fist', 2: 'ok', 3: 'three', 4: 'like', 5: 'call', 6: 'rock', 7: 'three2'}


In [6]:
onnx_path = model.export(
    format="onnx",
    imgsz=IMG_SIZE,
    simplify=True,
    nms=True,
    opset=12,
    dynamic=False,
)

print(f"\nONNX 변환 완료: {onnx_path}")

Ultralytics 8.4.71 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
WARNING ⚠️ 'nms=True' is not available for end2end models. Forcing 'nms=False'.
YOLOv10n summary (fused): 102 layers, 2,266,728 parameters, 0 gradients, 6.5 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/HandSpark/weights/hagrid_stage1_8cls_best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (5.5 MB)

ONNX: starting export with onnx 1.22.0 opset 12...
ONNX: slimming with onnxslim 0.1.94...
ONNX: export success ✅ 1.7s, saved as '/content/drive/MyDrive/HandSpark/weights/hagrid_stage1_8cls_best.onnx' (8.9 MB)

Export complete (3.2s)
Results saved to /content/drive/MyDrive/HandSpark/weights/hagrid_stage1_8cls_best.onnx
Predict:         yolo predict task=detect model=/content/drive/MyDrive/HandSpark/weights/hagrid_stage1_8cls_best.onnx imgsz=640 
Validate:        yolo val task=detect model=/content/drive/MyDrive/HandSpark/weights/hagrid_stage1_8cls_best.onnx imgsz=640 data

## 3. 변환 결과 확인

입력/출력 텐서 형태(shape)를 확인

In [7]:
import onnx

onnx_model = onnx.load(onnx_path)

print("=== 입력 정보 ===")
for inp in onnx_model.graph.input:
    dims = [d.dim_value if d.dim_value > 0 else d.dim_param for d in inp.type.tensor_type.shape.dim]
    print(f"  이름: {inp.name}, shape: {dims}")

print("\n=== 출력 정보 ===")
for out in onnx_model.graph.output:
    dims = [d.dim_value if d.dim_value > 0 else d.dim_param for d in out.type.tensor_type.shape.dim]
    print(f"  이름: {out.name}, shape: {dims}")

print("\n=== 클래스 목록 (인덱스 순서) ===")
for idx, name in model.names.items():
    print(f"  {idx}: {name}")

=== 입력 정보 ===
  이름: images, shape: [1, 3, 640, 640]

=== 출력 정보 ===
  이름: output0, shape: [1, 300, 6]

=== 클래스 목록 (인덱스 순서) ===
  0: palm
  1: fist
  2: ok
  3: three
  4: like
  5: call
  6: rock
  7: three2


## 4. ONNX Runtime으로 변환 검증

In [8]:
import onnxruntime as ort
import numpy as np
import cv2
import glob

session = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])

input_name = session.get_inputs()[0].name
output_names = [o.name for o in session.get_outputs()]
print("입력 이름:", input_name)
print("출력 이름:", output_names)

입력 이름: images
출력 이름: ['output0']


In [12]:
# 테스트 이미지 하나 준비 (이전 학습에서 쓴 테스트셋 중 하나, 경로 없으면 임의 이미지로 대체 처리함)
TEST_IMAGE_GLOB = "/content/data/hagrid_yolo(three_gun 없는 버전)/*/images/test/*.jpg"
test_imgs = glob.glob(TEST_IMAGE_GLOB)

if not test_imgs:
    print("테스트 이미지를 찾지 못함. 경로 직접 지정")

img_path = "/content/drive/MyDrive/HandSpark/thumb_001.PNG"
print(f"테스트 이미지: {img_path}")

img = cv2.imread(img_path)
img_resized = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
img_rgb = cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)

# 전처리: 0~1 정규화, HWC -> CHW, 배치 차원 추가
input_tensor = img_rgb.astype(np.float32) / 255.0
input_tensor = np.transpose(input_tensor, (2, 0, 1))
input_tensor = np.expand_dims(input_tensor, axis=0)

outputs = session.run(output_names, {input_name: input_tensor})

print("\n출력 shape:")
for name, out in zip(output_names, outputs):
    print(f"  {name}: {out.shape}")

print("\n출력 값 (앞부분 미리보기):")
print(outputs[0][:1])  # 첫 배치만 미리보기

테스트 이미지를 찾지 못함. 경로 직접 지정
테스트 이미지: /content/drive/MyDrive/HandSpark/thumb_001.PNG

출력 shape:
  output0: (1, 300, 6)

출력 값 (앞부분 미리보기):
[[[    0.11346      479.72      26.668      624.81   0.0036487           5]
  [    0.11346      479.72      26.668      624.81   0.0024725           4]
  [   0.054157      462.55       24.19      622.96   0.0011386           5]
  ...
  [     56.878      525.77      215.67      639.79  3.2485e-06           4]
  [    -3.2204      525.68      139.07      618.31  3.2485e-06           7]
  [     154.02      616.84      229.85       639.9  3.1888e-06           6]]]


## 5. Google Drive 백업

In [13]:
import shutil
import os

os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
dst_path = f"{DRIVE_SAVE_DIR}/hagrid_stage1_8cls2.onnx"
shutil.copy(onnx_path, dst_path)

print(f"저장됨: {dst_path}")

저장됨: /content/drive/MyDrive/HandSpark/weights/hagrid_stage1_8cls2.onnx
